In [1]:
%pip install -U langchain-ollama

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.10.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached charset_normalizer-3.4.3-cp311-cp311-macosx_10_9_universal2.whl.metadata (36 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 

In [ ]:
# AGENT SETUP

#I will be using ollama models (gpt-oss)
#ollama models library - https://ollama.com/library
#-------setup-----
#brew install ollama
#brew services start ollama

#-----fetch llm model----
#ollama pull gpt-oss:20b or download from ollama chat https://ollama.com/download 

# AGENTS

#Supervisor Agent

In [ ]:
# SMART CONTRACTS

#DataManager
#DataEvaluator
#ReportMaker

In [2]:
%pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 8.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import random
import numpy as np

# ---- contract stubs (replace with web3.py calls) ----
class DataManager:
    def execute(self, data):
        print("[DataManager] store:", data)
        return {"ok": True, "note": "stored"}

class DataEvaluator:
    def execute(self, data):
        print("[DataEvaluator] evaluate:", data)
        # pretend evaluation result
        return {"ok": True, "valid": True, "score": 0.87}

class ReportMaker:
    def execute(self, data):
        print("[ReportMaker] make report:", data)
        return {"ok": True, "report_id": "RPT-001"}

In [ ]:
#<------------------------------------------ SUPERVISOR AGNET ---------------------------------------------->

In [8]:
from typing import Dict, Any
import re
import json
import time

In [9]:
try:
    from ollama import chat, ChatResponse
    OLLAMA_PY_AVAILABLE = True
except Exception:
    OLLAMA_PY_AVAILABLE = False
    import requests

In [13]:




# ---- SupervisorAgent that uses gpt-oss ----
class SupervisorAgent:
    def __init__(self, model: str = "gpt-oss:20b", endpoint: str = "http://localhost:11434"):
        self.model = model
        self.endpoint = endpoint.rstrip("/")
        # contract map
        self.contracts = {
            "DataManager": DataManager(),
            "DataEvaluator": DataEvaluator(),
            "ReportMaker": ReportMaker()
        }
        # A compact prompt with few-shot mapping examples
        self.system_prompt = (
            "You are an automated Supervisor Agent. "
            "Given an ESG data record, reply ONLY with a JSON object with keys: "
            "'contract' (one of DataManager, DataEvaluator, ReportMaker), "
            "'reason' (short string). Example: {\"contract\":\"DataManager\",\"reason\":\"...\"}."
            "If unsure, pick DataEvaluator."
        )
        self.examples = [
            # example 1: raw reading -> store
            {
                "data": {"type": "sensor_reading", "metric": "CO2", "value": "320", "source": "factory_sensor"},
                "contract": "DataManager",
                "reason": "raw sensor reading should be stored first"
            },
            # example 2: suspicious value -> evaluate
            {
                "data": {"type": "report", "metric": "water_consumption", "value": "10000000", "source": "self_report"},
                "contract": "DataEvaluator",
                "reason": "unusually large self-reported value needs validation"
            },
            # example 3: aggregated evaluated data -> report
            {
                "data": {"type": "evaluated", "metric": "energy_use", "value": "5000", "source": "evaluator", "validated": True},
                "contract": "ReportMaker",
                "reason": "validated data ready for report generation"
            }
        ]

    # ---- helper: build prompt for the LLM ----
    def build_prompt(self, data: Dict[str, Any]) -> str:
        parts = [self.system_prompt, "\n\nExamples:\n"]
        for ex in self.examples:
            parts.append("DATA: " + json.dumps(ex["data"]))
            parts.append("DECISION: " + json.dumps({"contract": ex["contract"], "reason": ex["reason"]}))
            parts.append("")  # blank line
        parts.append("\nNow decide for this input:\nDATA: " + json.dumps(data))
        parts.append("DECISION:")
        return "\n".join(parts)

    

    def call_model_python(self, prompt: str, max_tokens: int = 200, temperature: float = 0.0):
        if not OLLAMA_PY_AVAILABLE:
            raise RuntimeError("Ollama python package not installed")
        resp: ChatResponse = chat(
            model=self.model,
            messages=[
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
            ],
            options={
                "num_predict": max_tokens,
                "temperature": temperature
            }
        )
        text = resp['message']['content']
        return text


    # ---- parse model response to pick contract ----
    def parse_response(self, text: str):
        # try to find a JSON object in the reply
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                j = json.loads(m.group(0))
                contract = j.get("contract")
                reason = j.get("reason", "")
                if contract in self.contracts:
                    return contract, reason
            except Exception:
                pass
        # fallback heuristics
        txt = text.lower()
        if "store" in txt or "save" in txt or "data manager" in txt:
            return "DataManager", text.strip()
        if "evaluate" in txt or "validate" in txt or "check" in txt:
            return "DataEvaluator", text.strip()
        if "report" in txt or "generate report" in txt:
            return "ReportMaker", text.strip()
        # default
        return "DataEvaluator", text.strip()

    # ---- public method to handle incoming data ----
    def handle_data(self, data: Dict[str, Any], use_rest: bool = False):
        prompt = self.build_prompt(data)
        # call the model
        if OLLAMA_PY_AVAILABLE and not use_rest:
            text = self.call_model_python(prompt)
        else:
            text = self.call_model_rest(prompt)
        contract_name, reason = self.parse_response(text)
        result = self.contracts[contract_name].execute(data)
        print(f"[Supervisor] decided: {contract_name} (reason: {reason}) -> result: {result}")
        return {"contract": contract_name, "reason": reason, "result": result, "raw_model": text}
    
    # ---- quick demo ----
if __name__ == "__main__":
    agent = SupervisorAgent()
    samples = [
        {"type":"sensor_reading","metric":"CO2","value":"320","source":"sensor-A"},
        {"type":"self_report","metric":"water","value":"10000000","source":"company-report"},
        {"type":"evaluated","metric":"energy","value":"5000","validated":True}
    ]
    for s in samples:
        out = agent.handle_data(s)
        time.sleep(0.5)


[DataManager] store: {'type': 'sensor_reading', 'metric': 'CO2', 'value': '320', 'source': 'sensor-A'}
[Supervisor] decided: DataManager (reason: raw sensor reading should be stored first) -> result: {'ok': True, 'note': 'stored'}
[DataEvaluator] evaluate: {'type': 'self_report', 'metric': 'water', 'value': '10000000', 'source': 'company-report'}
[Supervisor] decided: DataEvaluator (reason: self-reported value needs validation) -> result: {'ok': True, 'valid': True, 'score': 0.87}
[ReportMaker] make report: {'type': 'evaluated', 'metric': 'energy', 'value': '5000', 'validated': True}
[Supervisor] decided: ReportMaker (reason: validated data ready for report generation) -> result: {'ok': True, 'report_id': 'RPT-001'}
